# Assignment 4 - LoRA

Adapted from https://colab.research.google.com/github/huggingface/trl/blob/main/examples/notebooks/sft_trl_lora_qlora.ipynb?authuser=2

## Key concepts

- **SFT**: Trains models from example input-output pairs to align behavior with human preferences.
- **LoRA**: Updates only a few low-rank parameters, reducing training cost and memory.
- **QLoRA**: A quantized version of LoRA that enables even larger models to fit on small GPUs.
- **TRL**: The Hugging Face library that makes fine-tuning and reinforcement learning simple and efficient.

Learn how to perform **Supervised Fine-Tuning (SFT)** with **LoRA/QLoRA** using **TRL**.

## Install dependencies

We'll install **TRL** with the **PEFT** extra, which ensures all main dependencies such as **Transformers** and **PEFT** (a package for parameter-efficient fine-tuning, e.g., LoRA/QLoRA) are included. Additionally, we'll install **trackio** to log and monitor our experiments, and **bitsandbytes** to enable quantization of LLMs, reducing memory consumption for both inference and training.

In [ ]:
!pip install -Uq "trl[peft]" trackio bitsandbytes liger-kernel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.5/276.5 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 23.3 MB/s eta 0:00:00


### Log in to Hugging Face

Log in to your **Hugging Face** account to save your fine-tuned model, track your experiment results directly on the Hub or access gated models. You can find your **access token** on your [account settings page](https://huggingface.co/settings/tokens).

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Load Dataset

In this step, we load the [**dataset of your choice**](https://huggingface.co/datasets/) from the Hugging Face Hub using the `datasets` library.  

For efficiency, we'll load only the **training split**:

In [ ]:
from datasets import load_dataset

dataset_name = "HuggingFaceH4/deita-6k-v0-sft" # Dataset
train_dataset = load_dataset(dataset_name, split="train_sft")

README.md:   0%|          | 0.00/782 [00:00<?, ?B/s]

data/train_sft-00000-of-00001.parquet:   0%|          | 0.00/112M [00:00<?, ?B/s]

data/test_sft-00000-of-00001.parquet:   0%|          | 0.00/5.27M [00:00<?, ?B/s]

data/train_gen-00000-of-00001.parquet:   0%|          | 0.00/110M [00:00<?, ?B/s]

data/test_gen-00000-of-00001.parquet:   0%|          | 0.00/5.13M [00:00<?, ?B/s]

Generating train_sft split:   0%|          | 0/5700 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/300 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/5700 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/300 [00:00<?, ? examples/s]

Let's explore the training dataset.

In [ ]:
train_dataset

Dataset({
    features: ['prompt', 'prompt_id', 'messages'],
    num_rows: 5700
})

Let's see a full example to understand the internal structure:

In [ ]:
train_dataset[0]

{'prompt': 'I want to learn store Procedure, Can you teach me from the basic .',
 'prompt_id': '100cfc5f4d14b9c640cac30ebb28d0f91f2dc7662e9a578a544923a48fd5a59f',
 'messages': [{'content': 'I want to learn store Procedure, Can you teach me from the basic .',
   'role': 'user'},
  {'content': 'Of course! I\'d be happy to help you learn about stored procedures.\n\nA stored procedure is a set of pre-compiled SQL statements that are stored in a database, and can be executed on demand. Stored procedures can help simplify database programming by allowing you to write a piece of code once, and then reuse it multiple times.\n\nHere are some basic steps to get started with stored procedures:\n\n1. Create a stored procedure:\nTo create a stored procedure, you\'ll need to use a CREATE PROCEDURE statement. Here\'s an example:\n```sql\nCREATE PROCEDURE sp_GetCustomerDetails\n    @CustomerID int\nAS\nBEGIN\n    SELECT * FROM Customers\n    WHERE CustomerID = @CustomerID\nEND\n```\nIn this example, w

Now, let's remove any not required columns:

In [ ]:
train_dataset = train_dataset.remove_columns(column_names=['prompt', 'prompt_id'])

Convert question/query to messages

In [ ]:
#def format_to_messages(example):
 #   return {
  #      "messages": [
   #         {"role": "user", "content": example["question"]},
    #        {"role": "assistant", "content": example["query"]}
     #   ]
    #}

#train_dataset = train_dataset.map(format_to_messages)

In [ ]:
#def merge_thinking_and_remove_key(example):
 #   new_messages = []
  #  for msg in example["messages"]:
   #     content = msg["content"]
    #    thinking = msg.pop("thinking", None)
     #   if thinking and isinstance(thinking, str) and thinking.strip():
      #      content = f"<think>\n{thinking}\n</think>\n{content}"
       # msg["content"] = content
        #new_messages.append(msg)
    #example["messages"] = new_messages
    #return example

#train_dataset = train_dataset.map(merge_thinking_and_remove_key)

#def format_to_messages(example):
 #   return {
  #      "messages": [
   #         {"role": "user", "content": example["question"]},
    #        {"role": "assistant", "content": example["query"]}
     #   ]
    #}

#train_dataset = train_dataset.map(format_to_messages)

Map:   0%|          | 0/8168 [00:00<?, ? examples/s]

Map:   0%|          | 0/8168 [00:00<?, ? examples/s]

## Load model and configure LoRA/QLoRA

This notebook can be used with two fine-tuning methods. By default, it is set up for **QLoRA**, which includes quantization using `BitsAndBytesConfig`. If you prefer to use standard **LoRA** without quantization, simply comment out the `BitsAndBytesConfig` configuration.

Below, choose your **preferred model**. All of the options have been tested on **free Colab instances**.

**Note** that the larger models *may* cause problems in terms of running via a free Colab instance on the later steps. You may want to pick a smaller model if you're finding you're getting out of memory errors.

In [ ]:
# Select one model below by uncommenting the line you want to use
## Qwen
#model_id, output_dir = "unsloth/qwen3-14b-unsloth-bnb-4bit", "qwen3-14b-unsloth-bnb-4bit-SFT"     # ⚠️ ~14.1 GB VRAM
# model_id, output_dir = "Qwen/Qwen3-8B", "Qwen3-8B-SFT"                                          # ⚠️ ~12.8 GB VRAM
#model_id, output_dir = "Qwen/Qwen2.5-7B-Instruct", "Qwen2.5-7B-Instruct"                        # ✅ ~10.8 GB VRAM

## Llama
#model_id, output_dir = "meta-llama/Llama-3.2-3B-Instruct", "Llama-3.2-3B-Instruct"              # ✅ ~4.7 GB VRAM
# model_id, output_dir = "meta-llama/Llama-3.1-8B-Instruct", "Llama-3.1-8B-Instruct"              # ⚠️ ~10.9 GB VRAM

## Gemma
# model_id, output_dir = "google/gemma-3n-E2B-it", "gemma-3n-E2B-it"                              # ❌ Upgrade to a higher tier of colab
# model_id, output_dir = "google/gemma-3-4b-it", "gemma-3-4b-it"                                  # ⚠️ ~6.8 GB VRAM

## Granite
#model_id, output_dir = "ibm-granite/granite-4.0-micro", "granite-4.0-micro"                      # ✅ ~3.3 GB VRAM

## LFM2
model_id, output_dir = "LiquidAI/LFM2-2.6B", "LFM2-2.6B-SFT"                                     # ✅ ~5.89 GB VRAM

Let's load the selected model using `transformers`, configuring QLoRA via `bitsandbytes` (you can remove it if doing LoRA). We don't need to configure the tokenizer since the trainer takes care of that automatically.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
    quantization_config=bnb_config
)

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/5.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

The following cell defines LoRA (or QLoRA if needed). When training with LoRA/QLoRA, we use a **base model** (the one selected above) and, instead of modifying its original weights, we fine-tune a **LoRA adapter** — a lightweight layer that enables efficient and memory-friendly training. The **`target_modules`** specify which parts of the model (e.g., attention or projection layers) will be adapted by LoRA during fine-tuning.

In [ ]:
from peft import LoraConfig

# You may need to update `target_modules` depending on the architecture of your chosen model.
# For example, different LLMs might have different attention/projection layer names.
peft_config = LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
)

## Train model

We'll configure **SFT** using `SFTConfig`, keeping the parameters minimal so the training fits on a free Colab instance. You can adjust these settings if more resources are available. For full details on all available parameters, check the [TRL SFTConfig documentation](https://huggingface.co/docs/trl/sft_trainer#trl.SFTConfig).

In [ ]:
!pip install trl

In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    # Training schedule / optimization
    per_device_train_batch_size = 1,      # Batch size per GPU
    gradient_accumulation_steps = 4,      # Gradients are accumulated over multiple steps → effective batch size = 2 * 8 = 16
    warmup_steps = 5,
    # num_train_epochs = 1,               # Number of full dataset passes. For shorter training, use `max_steps` instead (this case)
    max_steps = 32,
    learning_rate = 4e-4,                 # Learning rate for the optimizer
    optim = "paged_adamw_8bit",           # Optimizer

    # Logging / reporting
    logging_steps=1,                      # Log training metrics every N steps
    report_to="none",                  # Experiment tracking tool
    trackio_space_id=output_dir,          # HF Space where the experiment tracking will be saved
    output_dir=output_dir,                # Where to save model checkpoints and logs

    max_length=1024,                      # Maximum input sequence length
    use_liger_kernel=True,                # Enable Liger kernel optimizations for faster training
    activation_offloading=True,           # Offload activations to CPU to reduce GPU memory usage

    # Hub integration
    push_to_hub=True,                     # Automatically push the trained model to the Hugging Face Hub
                                          # The model will be saved under your Hub account in the repository named `output_dir`

)

Configure the SFT Trainer. We pass the previously configured `training_args`. We don't use eval dataset to maintain memory usage low but you can configure it.

In [ ]:
from trl import SFTTrainer
print(train_dataset.column_names)
def format_example(example):
    return {"text": example["messages"]}

train_dataset = train_dataset.map(format_example)

#!pip install liger-kernel

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config,

)


['messages', 'text']


Map:   0%|          | 0/5700 [00:00<?, ? examples/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Tokenizing train dataset:   0%|          | 0/5700 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5700 [00:00<?, ? examples/s]

Show memory stats before training

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
1.602 GB of memory reserved.


And train! Note this may take up to 90 minutes depending on your settings.

In [ ]:
trainer_stats = trainer.train()

/usr/local/lib/python3.12/dist-packages/transformers/trainer.py:3809: UserWarning: liger-kernel did not return token_accuracy when requested. The mean_token_accuracy metric will not be logged. This is unexpected; please report it to the liger-kernel repository.
  loss = self.compute_loss(model, inputs, num_items_in_batch=num_items_in_batch)


Step,Training Loss
1,2.149807
2,2.144607
3,1.833933
4,2.775216
5,1.636395
6,1.169274
7,1.420965
8,2.749820
9,1.746811
10,1.743829


Show memory stats after training

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

962.5898 seconds used for training.
16.04 minutes used for training.
Peak reserved memory = 2.951 GB.
Peak reserved memory for training = 1.349 GB.
Peak reserved memory % of max memory = 20.264 %.
Peak reserved memory for training % of max memory = 9.263 %.


## Saving fine tuned model

In this step, we save the fine-tuned model both **locally** and to the **Hugging Face Hub** using the credentials from your account.

In [ ]:
trainer.save_model(output_dir)
trainer.push_to_hub(dataset_name=dataset_name)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ....6B-SFT/training_args.bin: 100%|##########| 5.58kB / 5.58kB            

  ...adapter_model.safetensors: 100%|##########| 4.73MB / 4.73MB            

No files have been modified since last commit. Skipping to prevent empty commit.


...

  ....6B-SFT/training_args.bin: 100%|##########| 5.58kB / 5.58kB            

  ....6B-SFT/training_args.bin: 100%|##########| 5.58kB / 5.58kB            

  ...adapter_model.safetensors: 100%|##########| 4.73MB / 4.73MB            

  ...adapter_model.safetensors: 100%|##########| 4.73MB / 4.73MB            

CommitInfo(commit_url='https://huggingface.co/tchakra1/LFM2-2.6B-SFT/commit/d2c31505aa71e605ae768642e8506c6fb893acfb', commit_message='End of training', commit_description='', oid='d2c31505aa71e605ae768642e8506c6fb893acfb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/tchakra1/LFM2-2.6B-SFT', endpoint='https://huggingface.co', repo_type='model', repo_id='tchakra1/LFM2-2.6B-SFT'), pr_revision=None, pr_num=None)

## Load the fine-tuned model and run inference

Now, let's test our fine-tuned model by loading the **LoRA/QLoRA adapter** and performing **inference**. We'll start by loading the **base model**, then attach the adapter to it, creating the final fine-tuned model ready for evaluation.

**NOTE**: You must replace 'guzdial' below with your own huggingface account name.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

adapter_model = f"tchakra1/{output_dir}" # Replace with your HF username or organization

# NOTE: if you have more GPU space from using a smaller model you can try the below line instead
#base_model = AutoModelForCausalLM.from_pretrained(model_id, dtype="float32", device_map="auto")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    attn_implementation="sdpa",                   # Change to Flash Attention if GPU has support
    dtype=torch.float16,                          # Change to bfloat16 if GPU has support
    use_cache=True,                               # Whether to cache attention outputs to speed up inference
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,                        # Load the model in 4-bit precision to save memory
        bnb_4bit_compute_dtype=torch.float16,     # Data type used for internal computations in quantization
        bnb_4bit_use_double_quant=True,           # Use double quantization to improve accuracy
        bnb_4bit_quant_type="nf4"                 # Type of quantization. "nf4" is recommended for recent LLMs
    )
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 42.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 23.81 MiB is free. Including non-PyTorch memory, this process has 14.54 GiB memory in use. Of the allocated memory 14.13 GiB is allocated by PyTorch, and 255.15 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

Here create a sample message using your dataset's structure. NOTE: You will need to do this multiple times.

In [ ]:

messages = [
  {
      "role": "system",
      "content": "You are a helpful assistant."
  },
  {
      "role": "user",
      "content":  "Why is data security important for modern organizations?",
  }
]

Let's first check what's the output for the base model, without the adapter.

In [ ]:
text = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=False
)
model_inputs = tokenizer([text], return_tensors="pt").to(base_model.device)

generated_ids = base_model.generate(
    **model_inputs,
    max_new_tokens=512
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]

# Decode and extract model response
generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
print(generated_text)

Time management is essential for increasing productivity and reducing stress. Here are some common techniques used to manage time effectively:

1. **Prioritization**: Use methods like the Eisenhower Matrix to categorize tasks by urgency and importance, focusing on what's truly important.

2. **To-Do Lists**: Create daily or weekly lists of tasks to keep track of responsibilities and deadlines.

3. **Time Blocking**: Allocate specific blocks of time for different tasks or activities throughout the day to ensure focused work periods.

4. **Pomodoro Technique**: Work in focused intervals (usually 25 minutes) followed by short breaks to maintain concentration and avoid burnout.

5. **Setting SMART Goals**: Define Specific, Measurable, Achievable, Relevant, and Time-bound goals to provide clear direction and motivation.

6. **Delegation**: Assign tasks to others when possible to free up time for more critical responsibilities.

7. **Avoiding Multitasking**: Focus on one task at a time to im

Did the base model meet your expectations? Let's now load the fine-tuned model and check its answer.

In [ ]:
fine_tuned_model = PeftModel.from_pretrained(base_model, adapter_model)

In [ ]:
text = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=False
)
model_inputs = tokenizer([text], return_tensors="pt").to(fine_tuned_model.device)

generated_ids = fine_tuned_model.generate(
    **model_inputs,
    max_new_tokens=512
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]

# Decode and extract model response
generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
print(generated_text)

Time management is a crucial skill that can help you stay organized, productive, and reduce stress. Here are some common techniques used in time management:

1. **Prioritization**: Identify the most important tasks and focus on them first. Use the Eisenhower Matrix to categorize tasks into four quadrants: urgent and important, important but not urgent, urgent but not important, and neither urgent nor important.

2. **Goal Setting**: Set specific, measurable, achievable, relevant, and time-bound (SMART) goals. Break down larger goals into smaller, manageable tasks.

3. **Time Blocking**: Allocate specific time slots for different tasks or activities. This helps you stay focused and avoid multitasking.

4. **Pomodoro Technique**: Work for a set amount of time (usually 25 minutes), then take a short break. After four cycles, take a longer break.

5. **Delegation**: If possible, delegate tasks to others. This can free up your time for more important tasks.

6. **Avoid Procrastination**: Br

If everything went well it should be different! If not, things may have gone wrong.

## Inference and Serving with vLLM

You can use Transformer models with **vLLM** to serve them in real-world applications. Learn more [here](https://blog.vllm.ai/2025/04/11/transformers-backend.html).

In [ ]:
!pip install -qU vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.9/432.9 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.9/34.9 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/1

### Push Merged Model (for LoRA or QLoRA Training)

To serve the model via **vLLM**, the repository must contain the merged model (base model + LoRA adapter). Therefore, you need to upload it first.

In [ ]:
model_merged = fine_tuned_model.merge_and_unload()

save_dir = f"{output_dir}-merged"

model_merged.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

In [ ]:
model_merged.push_to_hub(f"sergiopaniego/{output_dir}-merged") # Replace with your HF username or organization
tokenizer.push_to_hub(f"sergiopaniego/{output_dir}-merged") # Replace with your HF username or organization

### Performing Inference with vLLM

Use **vLLM** to run your model and generate text efficiently in real-time. This allows you to test and deploy your fine-tuned models with low latency and high throughput.

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
import torch

llm = LLM(
    model=f"guzdial/{output_dir}-merged", # Replace with your HF username or organization
    model_impl="transformers",                  # Select the transformers model implementation
    max_model_len=512,                         # Reduced for efficiency
    dtype=torch.float16
)
hf_tokenizer = AutoTokenizer.from_pretrained(f"guzdial/{output_dir}-merged")  # Replace with your HF username or organization

In [ ]:
# Alternatively, use llm.chat()
prompt = hf_tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

outputs = llm.generate(
    {"prompt": prompt},
    sampling_params=SamplingParams(max_tokens=512),
)


for o in outputs:
    generated_text = o.outputs[0].text
    print(generated_text)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

<think>
Mag nachdenken...igkeit. Ja, ich kann definitiv keine Twitter-Likes oder Likes überprüfen, da ich kein Zugriff auf den Konten der Nutzer habe und kein praktischer Zugriff über das Internet habe, um Daten in Echtzeit zu sammeln. Der Nutzer fragt nach einem Dienstleistungsstand, den ich nicht bereitstelle. Ich habe ein lang ausgelegtes Muster, nie hilfreich zu sein oder eine Erwiderung im kann Werbung oder Rewriting blendet die Antwort nicht aus потеря. Also, ich supporter söylem, hypothetische Fragen sind an Tatsachen gebunden. Ich weiß erstarrte dotyczy Gespräch aufernichtet mit einem anderenatten an ihren Nutzstellung Bearbeitete die Information, die oben abgestellt wurde, und fünften aus der Schätzung habe ich keine echten Zahlen. Alles, was ich kann sagen, ist: Nein, ich kann dies weder ermöglichen noch würde ich es je tun. In dem Sinne, 然后 ich wähle vor der Available antwortem, remains in das 'No' Verkleidung an,optiґxt; Alles, was ich zum Eintritt in den Band Emblem curve,